# T16 - 企业预警通地方化债数据爬虫

## 任务概述

企业预警通地方化债数据爬虫，用于爬取企业预警通(qyyjt.cn)网站的地方化债相关数据，并进行数据分析处理。

## 主要功能

- 爬取地方化债相关新闻和数据
- 支持无限滚动加载
- 详情页爬取
- 数据分析与可视化
- 规避反爬机制（418错误处理）

## 1. 环境配置

In [ ]:
# 安装依赖（如果需要）
# !pip install -r requirements.txt

In [ ]:
import sys
import os
from pathlib import Path

# 确保项目根目录在路径中
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"项目根目录: {project_root}")

In [ ]:
# 导入模块
import config
from utils import setup_logger, load_cookie_from_file
from main import DebtDataCrawler
from detail_crawler import DetailCrawler
from analyzer import DebtDataAnalyzer
from debt_trend_analyzer import DebtTrendAnalyzer

# 设置日志
logger = setup_logger()
logger.info("模块加载完成")

## 2. 配置检查

In [ ]:
# 检查环境变量配置
print("=== 配置信息 ===")
print(f"基础URL: {config.BASE_URL}")
print(f"API URL: {config.API_URL}")
print(f"详情页API: {config.DETAIL_API_URL}")
print(f"数据目录: {config.DATA_DIR}")
print(f"输出文件: {config.OUTPUT_FILE}")
print(f"日志文件: {config.LOG_FILE}")
print(f"爬取延迟: {config.CRAWL_DELAY}秒")
print(f"最大页数: {config.MAX_PAGES}")
print(f"增量爬取: {config.INCREMENTAL}")
print(f"无头模式: {config.HEADLESS}")

# 检查Cookie配置
cookie = load_cookie_from_file()
if cookie:
    print(f"Cookie状态: 已从文件加载 ({cookie[:30]}...)")
elif config.DEFAULT_COOKIE:
    print(f"Cookie状态: 已从环境变量加载 ({config.DEFAULT_COOKIE[:30]}...)")
else:
    print("Cookie状态: 未配置 - 请设置环境变量QYYJT_COOKIE或创建cookie.txt文件")

## 3. 数据爬取

### 3.1 爬取新闻列表

In [ ]:
# 初始化爬虫
crawler = DebtDataCrawler()

# 运行爬虫
crawler.run()

### 3.2 爬取详情页

In [ ]:
# 检查是否有列表数据
if config.OUTPUT_FILE.exists():
    detail_crawler = DetailCrawler()
    detail_crawler.run()
else:
    print("请先运行列表爬取")

## 4. 数据分析

### 4.1 基础数据分析

In [ ]:
# 加载数据
import pandas as pd

if config.OUTPUT_FILE.exists():
    df = pd.read_csv(config.OUTPUT_FILE, encoding='utf-8-sig')
    print(f"加载数据: {len(df)} 条")
    print(f"\n数据列: {list(df.columns)}")
    print(f"\n前5条数据:")
    df.head()
else:
    print("数据文件不存在，请先运行爬虫")

In [ ]:
# 运行基础分析
if config.OUTPUT_FILE.exists():
    analyzer = DebtDataAnalyzer()
    analyzer.run()

### 4.2 债务化解趋势分析

In [ ]:
# 运行趋势分析
if config.OUTPUT_FILE.exists():
    trend_analyzer = DebtTrendAnalyzer()
    trend_analyzer.run()

## 5. 查看分析结果

In [ ]:
# 列出分析结果文件
result_dir = config.DATA_DIR / "分析结果"
if result_dir.exists():
    print("=== 分析结果文件 ===")
    for file in result_dir.iterdir():
        print(f"  - {file.name}")
else:
    print("分析结果目录不存在")

In [ ]:
# 显示分析报告
report_file = config.DATA_DIR / "分析结果" / "分析报告.md"
if report_file.exists():
    with open(report_file, 'r', encoding='utf-8') as f:
        print(f.read())

In [ ]:
# 显示趋势分析报告
trend_report_file = config.DATA_DIR / "分析结果" / "地方债务化解趋势分析报告.md"
if trend_report_file.exists():
    with open(trend_report_file, 'r', encoding='utf-8') as f:
        print(f.read())

## 6. 显示可视化图表

In [ ]:
from IPython.display import Image, display

# 显示地区分布图
region_chart = config.DATA_DIR / "分析结果" / "地区分布.png"
if region_chart.exists():
    print("地区分布图:")
    display(Image(filename=str(region_chart)))

In [ ]:
# 显示时间趋势图
time_chart = config.DATA_DIR / "分析结果" / "时间趋势.png"
if time_chart.exists():
    print("时间趋势图:")
    display(Image(filename=str(time_chart)))

In [ ]:
# 显示关键词图
keyword_chart = config.DATA_DIR / "分析结果" / "关键词.png"
if keyword_chart.exists():
    print("关键词图:")
    display(Image(filename=str(keyword_chart)))

In [ ]:
# 显示债务化解方式图
method_chart = config.DATA_DIR / "分析结果" / "债务化解方式.png"
if method_chart.exists():
    print("债务化解方式图:")
    display(Image(filename=str(method_chart)))

In [ ]:
# 显示风险等级变化图
risk_chart = config.DATA_DIR / "分析结果" / "风险等级变化.png"
if risk_chart.exists():
    print("风险等级变化图:")
    display(Image(filename=str(risk_chart)))

In [ ]:
# 显示省份债务化解情况图
province_chart = config.DATA_DIR / "分析结果" / "省份债务化解情况.png"
if province_chart.exists():
    print("省份债务化解情况图:")
    display(Image(filename=str(province_chart)))

## 7. 使用说明

### 命令行使用

```bash
# 完整运行
python run.py

# 仅爬取列表
python run.py --mode crawl

# 仅爬取详情
python run.py --mode detail

# 仅分析
python run.py --mode analyze

# 增量更新
python run.py --incremental

# 使用自定义Cookie
python run.py --cookie "your_cookie_here"

# 慢速爬取
python run.py --slow
```

### 环境变量配置

复制 `.env.example` 为 `.env` 并填写配置：

```
QYYJT_COOKIE=your_cookie_here
QYYJT_CRAWL_DELAY=3
QYYJT_MAX_PAGES=300
```

### 注意事项

1. 合理设置爬取间隔，避免对网站造成压力
2. 数据仅供学习研究，请勿商业使用
3. 遵守网站robots.txt规则
4. 定期更新Cookie避免418错误